In [ ]:
%load_ext autoreload
%autoreload 2

# DP3 GainCal Python Binding

In [ ]:
import os
import shutil
import dp3
import numpy as np
import xarray as xr
import tempfile
import h5py
import dask
from distributed import Client

from ska_sdp_func_python.visibility.operations import expand_polarizations
from ska_sdp_datamodels.calibration import GainTable
from ska_sdp_datamodels.visibility import Visibility

from ska_sdp_instrumental_calibration.data_managers.solution_interval import (
    SolutionIntervals,
)
from ska_sdp_instrumental_calibration.numpy_processors.solvers.solver import Solver
from ska_sdp_instrumental_calibration.xarray_processors.solver import run_solver

from notebook_utils import compare_arrays, create_gaintable_from_h5param

In [ ]:
client = Client("tcp://127.0.0.1:34567")

In [ ]:
# Modify as required
ms_path = "/home/ska/Work/data/INST/lg3/cal_bpp_vis-lg3-rotated.small.ms"
dp3_skymodel_path = "./dp3_skymodel.txt"

gleam_skymodel_path = "/home/ska/Work/data/INST/sim/gleamegc.dat"

inst_output_path = "./inst_output_path"
dp3_output_path = "./dp3_gain_output"

inst_cache_dir = "/home/maneesh/Work/SKAO/ska-sdp-instrumental-calibration/cache"

In [ ]:
def clear_and_create_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.mkdir(path)

for path in [inst_output_path, dp3_output_path]:
    clear_and_create_dir(path)

In [ ]:
parset_file = f"{dp3_output_path}/gaincal.parset"
msout_path = f"{dp3_output_path}/dp3_output.ms"
dp3_output_gainpath = f"{dp3_output_path}/gaincal_via_execute.h5parm"

In [ ]:
from ska_sdp_instrumental_calibration.stages.load_data import load_data_stage
from ska_sdp_instrumental_calibration.stages.predict_visibilities import predict_visibilities
from ska_sdp_instrumental_calibration.scheduler import UpstreamOutput

up_out = UpstreamOutput()
input_ms_as_list = [ms_path]

up_out = load_data_stage(up_out, inst_output_path, input_ms_as_list, cache_directory=inst_cache_dir)

# Set just for convinience of running further cells
input_vis = up_out[0].vis
init_gaintable = up_out[0].gaintable

## DP3 via execute

In [ ]:
print("==============================================")
print(" Running DP3 C++ GainCal from Python")
print("==============================================")

parset_content = f"""
msin = "{ms_path}"
msin.datacolumn = DATA
msin.chunksize = 500
msout = "{msout_path}"

steps = [cal]

cal.type = gaincal
cal.sourcedb = "{dp3_skymodel_path}"
cal.parmdb = "{dp3_output_gainpath}"
cal.caltype = fulljones
cal.solint = 0
cal.nchan = 1
"""

with open(parset_file, "w") as file:
    file.write(parset_content)

print(f"Parset: {parset_file}")

# Python -> pybind11 -> DP3 C++
dp3.execute(parset_file, [])

print()
print("DP3 GainCal execution completed.")

## DP3 via in-memory python bindings 

In [ ]:
import threading
from dataclasses import dataclass

# DP3 is not safe to drive from multiple threads of one process:
# the gaincal solver uses DP3's process-wide (aocommon) thread pool, and
# the bundled HDF5 used to write the h5parm is built without threadsafety.
# Concurrent steps crash the process (std::bad_function_call / SIGABRT),
# e.g. on a dask worker with threads_per_worker > 1.
_DP3_LOCK_ATTR = "_inst_process_lock"


def _get_dp3_lock() -> threading.Lock:
    """
    Get the process-wide lock used to serialise DP3 calls.

    The lock is stored on the ``dp3`` module rather than as a global of
    this module, so that functions shipped to dask workers by value
    (cloudpickle) do not try to pickle a lock, and every worker process
    shares a single lock across its threads.

    Returns
    -------
    threading.Lock
        Lock shared by all threads of the current process.
    """
    import dp3  # pylint: disable=import-error,import-outside-toplevel

    # dict.setdefault is atomic under the GIL
    return dp3.__dict__.setdefault(_DP3_LOCK_ATTR, threading.Lock())


def _default_dp3_n_threads() -> int:
    """
    Default number of threads for DP3's internal thread pool.

    DP3 sizes its thread pool from the CPU affinity of the process, and
    ignores ``OMP_NUM_THREADS`` and threadpoolctl. On a dask worker this
    would make every worker use all cores, so the pool is limited to the
    number of threads of the worker instead. As DP3 calls are serialised
    by :py:func:`_get_dp3_lock`, a single DP3 call can use all of them.

    Returns
    -------
    int
        Number of threads of the current dask worker, or the number of
        CPUs available to the process when not running on a worker.
    """
    from distributed import (  # pylint: disable=import-outside-toplevel
        get_worker,
    )

    try:
        return get_worker().state.nthreads
    except ValueError:
        return len(os.sched_getaffinity(0))

_DP3_MODEL_DATA_NAME = "modeldata"
_DP3_POL_TO_RECEPTOR = {
    "XX": (0, 0),
    "XY": (0, 1),
    "YX": (1, 0),
    "YY": (1, 1),
}
_DP3_SUPPORTED_CALTYPES = (
    "scalarphase",
    "diagonal",
    "diagonalamplitude",
    "diagonalphase",
    "fulljones",
)


@dataclass
class DP3ObservationInfo:
    """
    Observation metadata required by DP3 to predict model visibilities.

    Parameters
    ----------
    antenna_names
        Names of the antennas. Shape: (nant,).
    antenna_positions
        ITRF XYZ antenna positions in meters. Shape: (nant, 3).
    antenna_diameters
        Antenna diameters in meters. Shape: (nant,).
    frequency
        Channel centre frequencies in Hz. Shape: (nfreq,).
    channel_bandwidth
        Channel widths in Hz. Shape: (nfreq,).
    time
        Centroid time of each timeslot in MJD seconds. Shape: (ntime,).
    integration_time
        Time interval between two timeslots in seconds.
    uvw
        Baseline UVW coordinates in meters. Shape: (ntime, nbl, 3).
    phasecentre
        Phase centre (ra, dec) in radians.
    """

    antenna_names: list[str]
    antenna_positions: np.ndarray
    antenna_diameters: np.ndarray
    frequency: np.ndarray
    channel_bandwidth: np.ndarray
    time: np.ndarray
    integration_time: float
    uvw: np.ndarray
    phasecentre: tuple[float, float]

    @classmethod
    def from_visibility(cls, vis: Visibility) -> "DP3ObservationInfo":
        """
        Create observation metadata from a Visibility.

        Parameters
        ----------
        vis
            Visibility to extract the metadata from.

        Returns
        -------
        DP3ObservationInfo
            Observation metadata of the visibility.
        """
        return cls(
            antenna_names=list(vis.configuration.names.data),
            antenna_positions=vis.configuration.xyz.data,
            antenna_diameters=vis.configuration.diameter.data,
            frequency=vis.frequency.data,
            channel_bandwidth=vis.channel_bandwidth.data,
            time=vis.time.data,
            integration_time=float(vis.integration_time.data[0]),
            uvw=vis.uvw.data,
            phasecentre=(vis.phasecentre.ra.rad, vis.phasecentre.dec.rad),
        )


def _n_per_interval(n_samples: int, n_intervals: int, axis: str) -> int:
    """
    Number of samples per solution interval along one axis.

    Parameters
    ----------
    n_samples
        Number of visibility samples along the axis.
    n_intervals
        Number of solution intervals along the axis.
    axis
        Name of the axis, used in the error message.

    Returns
    -------
    int
        Number of samples per solution interval.

    Raises
    ------
    ValueError
        If the samples cannot be split into exactly ``n_intervals`` chunks.
    """
    n_per = int(np.ceil(n_samples / n_intervals))
    if int(np.ceil(n_samples / n_per)) != n_intervals:
        raise ValueError(
            f"dp3_gaincal: cannot split {n_samples} {axis} samples into "
            f"{n_intervals} solution intervals"
        )
    return n_per


def _create_dp_info(
    ant1: np.ndarray,
    ant2: np.ndarray,
    n_ant: int,
    n_time: int,
    n_freq: int,
    observation: DP3ObservationInfo | None,
):
    """
    Create the DP3 DPInfo describing the visibilities.

    Parameters
    ----------
    ant1
        Indices of antenna 1 for each baseline. Shape: (nbl,).
    ant2
        Indices of antenna 2 for each baseline. Shape: (nbl,).
    n_ant
        Number of antennas.
    n_time
        Number of timeslots.
    n_freq
        Number of frequency channels.
    observation
        Observation metadata. If None, placeholder metadata is used,
        which is only valid when DP3 does not predict the model.

    Returns
    -------
    dp3.DPInfo
        DPInfo with 4 correlations.
    """
    import dp3  # pylint: disable=import-error,import-outside-toplevel

    # gaincal only works with 4 correlations
    dpinfo = dp3.DPInfo(4)

    if observation is None:
        dpinfo.set_channels(np.arange(n_freq, dtype=float) + 1.0, np.ones(n_freq))
        dpinfo.set_antennas(
            [f"ANT{idx}" for idx in range(n_ant)],
            np.ones(n_ant),
            np.zeros((n_ant, 3)),
            ant1.astype(int),
            ant2.astype(int),
        )
        dpinfo.set_times(0.5, n_time - 0.5, 1.0)
        dpinfo.phase_center = [0.0, 0.0]
    else:
        dpinfo.set_channels(observation.frequency, observation.channel_bandwidth)
        dpinfo.set_antennas(
            observation.antenna_names,
            observation.antenna_diameters,
            observation.antenna_positions,
            ant1.astype(int),
            ant2.astype(int),
        )
        dpinfo.set_times(
            observation.time[0],
            observation.time[-1],
            observation.integration_time,
        )
        dpinfo.phase_center = list(observation.phasecentre)

    return dpinfo


def _read_dp3_soltab(
    soltab: h5py.Group,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Read a DP3 h5parm soltab into Jones-matrix shaped arrays.

    Parameters
    ----------
    soltab
        h5parm soltab group, e.g. ``sol000/amplitude000``.

    Returns
    -------
    tuple of np.ndarray
        ``(values, weights, solved)`` each of shape
        (time, ant, freq, 2, 2). ``solved`` is a boolean mask of the
        receptor pairs present in the soltab.
    """
    axes = soltab["val"].attrs["AXES"]
    if isinstance(axes, bytes):
        axes = axes.decode()
    axes = str(axes).split(",")

    order = ["time", "ant", "freq"] + (["pol"] if "pol" in axes else [])
    perm = [axes.index(ax) for ax in order]
    val = np.transpose(soltab["val"][...], perm)
    weight = np.transpose(soltab["weight"][...], perm)

    shape = (*val.shape[:3], 2, 2)
    out_val = np.zeros(shape, dtype=val.dtype)
    out_weight = np.zeros(shape, dtype=weight.dtype)
    solved = np.zeros((2, 2), dtype=bool)

    if "pol" in axes:
        pols = [p.decode().strip("\x00") for p in soltab["pol"][...]]
        for idx, pol in enumerate(pols):
            r1, r2 = _DP3_POL_TO_RECEPTOR[pol]
            out_val[..., r1, r2] = val[..., idx]
            out_weight[..., r1, r2] = weight[..., idx]
            solved[r1, r2] = True
    else:
        # scalar solutions apply equally to both receptors
        for r in range(2):
            out_val[..., r, r] = val
            out_weight[..., r, r] = weight
            solved[r, r] = True

    return out_val, out_weight, np.broadcast_to(solved, shape)

In [ ]:
def dp3_gaincal_solve(
    vis_vis: np.ndarray,
    vis_flags: np.ndarray,
    vis_weight: np.ndarray,
    model_vis: np.ndarray | None,
    model_flags: np.ndarray | None,
    gain_gain: np.ndarray,
    gain_weight: np.ndarray,
    gain_residual: np.ndarray,
    ant1: np.ndarray,
    ant2: np.ndarray,
    *,
    caltype: str = "diagonal",
    niter: int = 50,
    tol: float = 1e-6,
    skymodel_path: str | None = None,
    observation: DP3ObservationInfo | None = None,
    n_threads: int | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Solve for antenna gains using the DP3 gaincal step in memory.

    The visibilities are streamed one timeslot at a time through a DP3
    ``gaincal`` step created via the python bindings. If ``model_vis`` is
    given, it is supplied as extra buffer data (``gaincal.reusemodel``).
    Otherwise DP3 predicts the model internally from ``skymodel_path``
    (``gaincal.sourcedb``), which requires the ``observation`` metadata.
    The solution intervals in time and frequency are derived from the
    shape of ``gain_gain``.

    DP3 can only write solutions to an h5parm file, so the solutions are
    written to a temporary file and read back.

    Parameters
    ----------
    vis_vis
        Observed visibility data. Shape: (time, baseline, freq, pol).
    vis_flags
        Flags for observed visibilities. Shape matches `vis_vis`.
    vis_weight
        Weights for observed visibilities. Shape matches `vis_vis`.
    model_vis
        Model visibility data. Shape matches `vis_vis`.
        If None, DP3 predicts the model from ``skymodel_path``.
    model_flags
        Flags for model visibilities. Shape matches `vis_vis`.
        Flagged model samples are also flagged in the observed data.
    gain_gain
        Initial gain estimates. Shape: (time, ant, freq, rec1, rec2).
        Only the shape and the receptor terms not solved by DP3 for the
        given ``caltype`` are used; DP3 starts from its own initial guess.
    gain_weight
        Storage for gain weights. Shape matches `gain_gain`.
        Receptor terms not solved by DP3 get zero weight.
    gain_residual
        Storage for gain residuals. Shape matches `gain_gain`.
        DP3 does not report residuals, so this is returned unchanged.
    ant1
        Indices of antenna 1 for each baseline. Shape: (nbl,).
    ant2
        Indices of antenna 2 for each baseline. Shape: (nbl,).
    caltype
        DP3 gaincal calibration type, e.g. ``diagonal``, ``scalarphase``
        or ``fulljones``.
    niter
        Maximum number of solver iterations (``gaincal.maxiter``).
    tol
        Solver convergence tolerance (``gaincal.tolerance``).
    skymodel_path
        Path to the DP3 sky model used to predict the model visibilities.
        Required if ``model_vis`` is None, ignored otherwise.
    observation
        Observation metadata. Required if ``model_vis`` is None. If not
        given while ``model_vis`` is provided, placeholder metadata is
        used.
    n_threads
        Number of threads for DP3's internal thread pool. If None, the
        number of threads of the current dask worker is used, or all
        available CPUs when not running on a worker.

    Returns
    -------
    tuple of np.ndarray
        A tuple containing:

        - Updated gain array.
        - Gain weights.
        - Gain residuals.

    Raises
    ------
    ValueError
        If neither a model nor a sky model with observation metadata is
        provided, ``caltype`` is not supported or the gain shape is
        incompatible with the visibility shape.
    """
    import dp3  # pylint: disable=import-error,import-outside-toplevel
    from dp3.parameterset import (  # pylint: disable=import-error,import-outside-toplevel # noqa: E501
        ParameterSet,
    )

    if model_vis is None and (skymodel_path is None or observation is None):
        raise ValueError(
            "dp3_gaincal: skymodel_path and observation are required "
            "when model_vis is not provided"
        )
    if caltype not in _DP3_SUPPORTED_CALTYPES:
        raise ValueError(
            f"dp3_gaincal: unsupported caltype {caltype!r}. "
            f"Supported: {', '.join(_DP3_SUPPORTED_CALTYPES)}"
        )

    n_time, n_baseline, n_freq, _ = vis_vis.shape
    n_soln_time, n_ant, n_soln_freq, _, _ = gain_gain.shape
    solint = _n_per_interval(n_time, n_soln_time, "time")
    nchan = _n_per_interval(n_freq, n_soln_freq, "frequency")

    flags = vis_flags if model_flags is None else vis_flags | model_flags

    dpinfo = _create_dp_info(ant1, ant2, n_ant, n_time, n_freq, observation)

    with tempfile.TemporaryDirectory() as tmpdir:
        h5parm_path = os.path.join(tmpdir, "gaincal.h5")

        parset = ParameterSet()
        parset.add("gaincal.parmdb", h5parm_path)
        if model_vis is None:
            parset.add("gaincal.sourcedb", skymodel_path)
        else:
            parset.add("gaincal.reusemodel", _DP3_MODEL_DATA_NAME)
        parset.add("gaincal.caltype", caltype)
        parset.add("gaincal.solint", str(solint))
        parset.add("gaincal.nchan", str(nchan))
        parset.add("gaincal.maxiter", str(niter))
        parset.add("gaincal.tolerance", str(tol))

        # Serialise all DP3 calls within this process, see _get_dp3_lock
        with _get_dp3_lock():
            # DP3's thread pool is process-wide, so set it on every call
            dp3.set_n_threads(
                _default_dp3_n_threads() if n_threads is None else n_threads
            )
            step = dp3.make_step("gaincal", parset, "gaincal.", dp3.MsType.regular)
            step.set_next_step(
                dp3.make_step("null", ParameterSet(), "", dp3.MsType.regular)
            )
            step.set_info(dpinfo)

            zero_uvw = np.zeros((n_baseline, 3))
            for time_idx in range(n_time):
                dpbuffer = dp3.DPBuffer()
                if observation is None:
                    dpbuffer.set_time(time_idx + 0.5)
                    dpbuffer.set_uvw(zero_uvw)
                else:
                    dpbuffer.set_time(observation.time[time_idx])
                    # DP3 uses the opposite uvw sign convention
                    dpbuffer.set_uvw(-observation.uvw[time_idx])
                dpbuffer.set_data(expand_polarizations(vis_vis[time_idx], np.complex64))
                dpbuffer.set_weights(expand_polarizations(vis_weight[time_idx], np.float32))
                dpbuffer.set_flags(expand_polarizations(flags[time_idx], bool))
                if model_vis is not None:
                    dpbuffer.add_data(_DP3_MODEL_DATA_NAME)
                    dpbuffer.set_extra_data(
                        _DP3_MODEL_DATA_NAME,
                        expand_polarizations(model_vis[time_idx], np.complex64),
                    )
                step.process(dpbuffer)
            step.finish()

        with h5py.File(h5parm_path, "r") as h5f:
            solution = h5f["sol000"]
            amplitude = (
                _read_dp3_soltab(solution["amplitude000"])
                if "amplitude000" in solution
                else None
            )
            phase = (
                _read_dp3_soltab(solution["phase000"])
                if "phase000" in solution
                else None
            )

    if amplitude is not None and phase is not None:
        amp_val, weight, solved = amplitude
        phase_val = phase[0]
        weight = np.minimum(weight, phase[1])
    elif amplitude is not None:
        amp_val, weight, solved = amplitude
        phase_val = np.zeros_like(amp_val)
    else:
        phase_val, weight, solved = phase
        amp_val = np.ones_like(phase_val)

    _gain_gain = np.where(solved, amp_val * np.exp(1j * phase_val), gain_gain).astype(
        gain_gain.dtype
    )
    _gain_weight = np.where(solved, weight, gain_weight).astype(gain_weight.dtype)

    return _gain_gain, _gain_weight, gain_residual.copy()

### Computation

In [ ]:
input_vis_computed = input_vis.compute()
init_gaintable_computed = init_gaintable.compute()

In [ ]:
_gain, _weight, _residual = dp3_gaincal_solve(
    input_vis_computed.vis.data,
    input_vis_computed.flags.data,
    input_vis_computed.weight.data,
    None,
    None,
    init_gaintable_computed.gain.data,
    init_gaintable_computed.weight.data,
    init_gaintable_computed.residual.data,
    input_vis_computed.antenna1.data,
    input_vis_computed.antenna2.data,
    caltype="fulljones",
    skymodel_path=dp3_skymodel_path,
    observation=DP3ObservationInfo.from_visibility(input_vis),
)

In [ ]:
dp3_gaincal_gaintable = init_gaintable.copy(deep=True)

dp3_gaincal_gaintable.gain.data = _gain
dp3_gaincal_gaintable.weight.data = _weight
dp3_gaincal_gaintable.residual.data = _residual

## Comparision

In [ ]:
dp3_gaintable: GainTable = create_gaintable_from_h5param(
    dp3_output_gainpath,
    interval=SolutionIntervals(input_vis.time.data, "full").intervals,
    vis=input_vis,
)

In [ ]:
xr.testing.assert_allclose(dp3_gaincal_gaintable, dp3_gaintable)

## Calling Solver in Dask Distribution

In [ ]:
class Dp3GaincalSolver(Solver):
    """
    Solver for antenna gains using the DP3 gaincal step.

    This class wraps :py:func:`dp3_gaincal_solve`, which streams the
    visibilities through an in-memory DP3 ``gaincal`` step. The model
    visibilities are always provided to DP3 (``gaincal.reusemodel``), so
    no sky model prediction happens inside DP3.

    Parameters
    ----------
    caltype
        DP3 gaincal calibration type. One of ``scalarphase``,
        ``diagonal``, ``diagonalamplitude``, ``diagonalphase`` or
        ``fulljones``. Default is ``diagonal``.
    **kwargs
        Additional keyword arguments passed to the base `Solver` class
        (e.g., `niter`, `tol`). Unknown arguments are ignored.

    Attributes
    ----------
    caltype : str
        DP3 gaincal calibration type.

    Examples
    --------
    >>> solver = Solver.get_solver("dp3_gaincal", caltype="fulljones")
    >>> gains, wgt, resid = solver.solve(vis, flags, wgt, model, ...)
    """

    _SOLVER_NAME_ = "dp3_gaincal"

    def __init__(self, caltype: str = "diagonal", **kwargs):
        super(Dp3GaincalSolver, self).__init__(**kwargs)
        if caltype not in _DP3_SUPPORTED_CALTYPES:
            raise ValueError(
                f"dp3_gaincal: unsupported caltype {caltype!r}. "
                f"Supported: {', '.join(_DP3_SUPPORTED_CALTYPES)}"
            )
        self.caltype = caltype

    def solve(
        self,
        vis_vis: np.ndarray,
        vis_flags: np.ndarray,
        vis_weight: np.ndarray,
        model_vis: np.ndarray,
        model_flags: np.ndarray,
        gain_gain: np.ndarray,
        gain_weight: np.ndarray,
        gain_residual: np.ndarray,
        ant1: np.ndarray,
        ant2: np.ndarray,
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Run the DP3 gaincal solver.

        Parameters
        ----------
        vis_vis
            Complex observed visibilities. Shape: (ntime, nbl, nchan, npol).
        vis_flags
            Boolean flags for observed visibilities (True is flagged).
        vis_weight
            Weights for observed visibilities.
        model_vis
            Complex model visibilities. Shape matches `vis_vis`.
        model_flags
            Boolean flags for model visibilities.
        gain_gain
            Initial guess for complex gains. Shape:
            (ntime_sol, nant, nchan_sol, nrec, nrec).
        gain_weight
            Weights for the gain solutions.
        gain_residual
            Buffer to store residuals of the fit. Returned unchanged, as
            DP3 does not report residuals.
        ant1
            Indices of antenna 1 for each baseline.
        ant2
            Indices of antenna 2 for each baseline.

        Returns
        -------
        tuple of np.ndarray
            A tuple containing (gain_gain, gain_weight, gain_residual) with
            the updated solutions.

        Raises
        ------
        ValueError
            If `model_vis` is not provided.
        """
        if model_vis is None:
            raise ValueError("dp3_gaincal: model_vis must be provided")

        return dp3_gaincal_solve(
            vis_vis,
            vis_flags,
            vis_weight,
            model_vis,
            model_flags,
            gain_gain,
            gain_weight,
            gain_residual,
            ant1,
            ant2,
            caltype=self.caltype,
            niter=self.niter,
            tol=self.tol,
        )

## Aligned comparison: `run_solver` vs DP3 execute

`solver_gaintable` above uses INST's `predict_vis` model, while DP3 execute predicts its own model from `dp3_skymodel.txt`, so the two gaintables are not directly comparable.

To align them, we use a common model:
1. Copy the input MS and use DP3 `predict` to write the model into its `MODEL_DATA` column.
2. DP3 execute gaincal reads that column (`cal.usemodelcolumn` + `cal.modelcolumn`).
3. INST loads `DATA` and `MODEL_DATA` from the same MS as `vis` / `modelvis` for `run_solver`.

Both solvers then see identical data, model, flags and weights.

In [ ]:
# NOTE: `output_path` is overwritten by the INST cell above,
# so derive the DP3 output dir from the parset path.
aligned_dir = os.path.join(dp3_output_path, "aligned")
aligned_ms_path = f"{aligned_dir}/aligned.ms"
aligned_dp3_gainpath = f"{aligned_dir}/gaincal_via_execute.h5parm"

if os.path.exists(aligned_dir):
    shutil.rmtree(aligned_dir)
os.makedirs(aligned_dir)

# Work on a copy so that the input MS is never modified
shutil.copytree(ms_path, aligned_ms_path)

# Step 1: DP3 predict -> MODEL_DATA column of the copied MS (in place)
predict_parset = f"""
msin = "{aligned_ms_path}"
msin.datacolumn = DATA
msout = .
msout.datacolumn = MODEL_DATA

steps = [predict]

predict.type = predict
predict.sourcedb = "{dp3_skymodel_path}"
"""

predict_parset_file = f"{aligned_dir}/predict.parset"
with open(predict_parset_file, "w") as file:
    file.write(predict_parset)

dp3.execute(predict_parset_file, [])

In [ ]:
# Step 2: DP3 gaincal via execute, reading the model from MODEL_DATA
# instead of predicting it from the sourcedb.
# niter / tol are set to match the Solver used below.
aligned_gaincal_parset = f"""
msin = "{aligned_ms_path}"
msin.datacolumn = DATA
msout = ""

steps = [cal]

cal.type = gaincal
cal.usemodelcolumn = true
cal.modelcolumn = MODEL_DATA
cal.parmdb = "{aligned_dp3_gainpath}"
cal.caltype = fulljones
cal.solint = 0
cal.nchan = 1
cal.maxiter = 50
cal.tolerance = 1e-6
"""

aligned_gaincal_parset_file = f"{aligned_dir}/gaincal.parset"
with open(aligned_gaincal_parset_file, "w") as file:
    file.write(aligned_gaincal_parset)

dp3.execute(aligned_gaincal_parset_file, [])

In [ ]:
# INST: read DATA and MODEL_DATA of the same MS as two Visibility datasets.
# NOTE: the zarr cache key is "<ms>_fid<id>_ddid<id>" and does NOT include
# the datacolumn, so each column needs its own cache_directory.
aligned_up_out = load_data_stage(
    UpstreamOutput(),
    aligned_dir,
    [aligned_ms_path],
    datacolumn="DATA",
    cache_directory=f"{aligned_dir}/cache_data",
)[0]
aligned_model_up_out = load_data_stage(
    UpstreamOutput(),
    aligned_dir,
    [aligned_ms_path],
    datacolumn="MODEL_DATA",
    cache_directory=f"{aligned_dir}/cache_model",
)[0]

aligned_vis = aligned_up_out.vis
aligned_modelvis = aligned_model_up_out.vis

In [ ]:
# # Optional diagnostic: how far is INST's predicted model from DP3's model?
# # Non-zero differences here explain why the un-aligned comparison fails.

# def dump():
#     predict_up_out = predict_visibilities(up_out[0], inst_output_path, input_ms_as_list, gleamfile=gleam_skymodel_path, use_everybeam=False, fov=0.1)

#     compare_arrays(
#         predict_up_out.modelvis.vis.data[... , 3],
#         aligned_modelvis.vis.data[... , 3],
#         rtol=1e-5,
#         atol=1e-8,
#         meta="Modelvis (INST predict vs DP3 predict)",
#     )

# dump()

In [ ]:
solver = Solver.get_solver(
    "dp3_gaincal", caltype="fulljones", niter=50, tol=1e-6
)

In [ ]:
aligned_solver_gaintable = run_solver(
    aligned_vis,
    aligned_modelvis,
    aligned_up_out.gaintable,
    solver,
)

aligned_solver_gaintable_computed = dask.compute(
    aligned_solver_gaintable, optimize_graph=True
)[0]

In [ ]:
aligned_dp3_gaintable: GainTable = create_gaintable_from_h5param(
    aligned_dp3_gainpath,
    interval=SolutionIntervals(aligned_vis.time.data, "full").intervals,
    vis=aligned_vis,
)

In [ ]:
compare_arrays(
    aligned_solver_gaintable_computed.gain.data,
    aligned_dp3_gaintable.gain.data,
    meta="Gain (solver vs DP3 execute)",
)

In [ ]:
compare_arrays(
    aligned_solver_gaintable_computed.weight.data,
    aligned_dp3_gaintable.weight.data,
    meta="Weight (solver vs DP3 execute)",
)

## Running INST independently

For testing with larger dataset. Not running DP3 for comparision here

In [ ]:
def t():
    _ms_path = "/home/ska/Work/data/INST/lg3/cal_bpp_vis-lg3-rotated.ms"
    _inst_output_path = "./inst_independent_run/"
    clear_and_create_dir(_inst_output_path)

    _inst_up_out_ = UpstreamOutput()
    _input_ms_as_list = [_ms_path]
    _inst_up_out_ = load_data_stage(_inst_up_out_, _inst_output_path, _input_ms_as_list, cache_directory=inst_cache_dir)

    predict_up_out = predict_visibilities(_inst_up_out_[0], _inst_output_path, _input_ms_as_list, gleamfile=gleam_skymodel_path, use_everybeam=False, fov=0.1)

    _solver = Solver.get_solver(
        "dp3_gaincal", caltype="fulljones", niter=50, tol=1e-6
    )

    inst_output = run_solver(
        predict_up_out.vis,
        predict_up_out.modelvis,
        predict_up_out.gaintable,
        _solver,
    )

    inst_output_computed = dask.compute(
        inst_output, optimize_graph=True
    )[0]

t()